# Experiment 4: Recent Dataset External Stress Test

        Use this notebook instead of modifying the initial GenImage notebook.

        Purpose:

        - Keep the original GenImage notebook for closed-set training and temperature scaling.
        - Use this notebook only for the newest-dataset experiment.
        - Evaluate the saved GenImage-trained ResNet-50 detector on Defactify / MS COCOAI without fine-tuning.

        Dataset:

        - `Rajarshi-Roy-research/Defactify_Image_Dataset`
        - Real MS COCO images plus Stable Diffusion 2.1, SDXL, SD3, DALL-E 3, and Midjourney v6.

        Thesis interpretation:

        > This experiment tests whether a detector trained on the original GenImage distribution remains reliable on newer generator distributions.

In [ ]:
!pip -q install -U datasets pillow scikit-learn scipy pandas matplotlib tqdm hf_xet

In [ ]:
from pathlib import Path
        import io
        import json
        import os
        import re
        import shutil

        import numpy as np
        import pandas as pd
        import matplotlib.pyplot as plt
        from PIL import Image
        from tqdm.auto import tqdm
        from datasets import load_dataset, Image as HFImage

        import tensorflow as tf
        from scipy.special import expit, logit, softmax
        from sklearn.metrics import (
            accuracy_score,
            balanced_accuracy_score,
            brier_score_loss,
            confusion_matrix,
            log_loss,
            roc_auc_score,
        )

        try:
            from google.colab import files
            IN_COLAB = True
        except Exception:
            files = None
            IN_COLAB = False

        print("TensorFlow:", tf.__version__)
        print("GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
# -----------------------------
        # Configuration
        # -----------------------------

        DATASET_ID = "Rajarshi-Roy-research/Defactify_Image_Dataset"
        SPLIT_TO_USE = "test"

        # Start with 500 or 1000 per source for a fast run.
        # For final results, set MAX_PER_SOURCE = None if runtime/disk allows.
        MAX_PER_SOURCE = 500
        RANDOM_SEED = 42

        IMG_SIZE = 224
        BATCH_SIZE = 32

        # Your saved model from the original GenImage notebook.
        # If this file is missing in Colab, the notebook will ask you to upload it.
        MODEL_PATH = Path("/content/resnet50_finetuned.keras")

        # If you use Drive, uncomment these two lines:
        # from google.colab import drive; drive.mount("/content/drive")
        # MODEL_PATH = Path("/content/drive/MyDrive/thesis_resnet50/resnet50_finetuned.keras")

        # Use the temperature from your original GenImage calibration.
        # Set to None if you want uncalibrated results only.
        TEMPERATURE = 1.7894

        HIGH_CONF_THRESHOLD = 0.90
        OUTPUT_DIR = Path("/content/defactify_external_eval") if Path("/content").exists() else Path("./defactify_external_eval")
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

        SOURCE_MAP = {
            0: "Real MS COCO",
            1: "Stable Diffusion 2.1",
            2: "Stable Diffusion XL",
            3: "Stable Diffusion 3",
            4: "DALL-E 3",
            5: "Midjourney v6",
        }

        BINARY_MAP = {
            0: "Real",
            1: "AI-generated",
        }

        print("Output directory:", OUTPUT_DIR)

## Model File

        If you already saved the model from the initial notebook, either upload `resnet50_finetuned.keras` here or point `MODEL_PATH` to the Drive path.

In [ ]:
if not MODEL_PATH.exists():
            if IN_COLAB:
                print(f"Model not found at {MODEL_PATH}")
                print("Upload your saved model file: resnet50_finetuned.keras")
                uploaded = files.upload()
                if uploaded:
                    uploaded_name = next(iter(uploaded.keys()))
                    uploaded_path = Path("/content") / uploaded_name
                    if uploaded_path != MODEL_PATH:
                        shutil.move(str(uploaded_path), str(MODEL_PATH))
            else:
                raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

        if not MODEL_PATH.exists():
            raise FileNotFoundError(
                "Model file is still missing. Save/upload resnet50_finetuned.keras before running evaluation."
            )

        print("Using model:", MODEL_PATH)
        model = tf.keras.models.load_model(MODEL_PATH)
        model.summary()

## Load Defactify / MS COCOAI

        The default uses a balanced subset from the test split. This is enough for a first thesis run. For final full results, set `MAX_PER_SOURCE = None`.

In [ ]:
def find_column(columns, candidates):
            exact = {c: c for c in columns}
            lower = {c.lower(): c for c in columns}
            for candidate in candidates:
                if candidate in exact:
                    return exact[candidate]
                if candidate.lower() in lower:
                    return lower[candidate.lower()]
            raise KeyError(f"Could not find any of {candidates} in {columns}")


        print(f"Loading {DATASET_ID}, split={SPLIT_TO_USE!r}")
        ds = load_dataset(DATASET_ID, split=SPLIT_TO_USE)

        IMAGE_COL = find_column(ds.column_names, ["Image", "image"])
        CAPTION_COL = find_column(ds.column_names, ["Caption", "caption", "prompt", "text"])
        LABEL_COL = find_column(ds.column_names, ["Label_A", "label_a", "label", "binary_label"])
        SOURCE_COL = find_column(ds.column_names, ["Label_B", "label_b", "source", "source_label"])

        # Keep image bytes/paths until individual rows are used.
        ds = ds.cast_column(IMAGE_COL, HFImage(decode=False))

        print(ds)
        print("Columns:", ds.column_names)
        print("Features:", ds.features)
        print("Using:", IMAGE_COL, CAPTION_COL, LABEL_COL, SOURCE_COL)

In [ ]:
labels_a = np.asarray(ds[LABEL_COL], dtype=int)
        labels_b = np.asarray(ds[SOURCE_COL], dtype=int)

        source_counts = (
            pd.Series(labels_b)
            .value_counts()
            .sort_index()
            .rename_axis("source_id")
            .reset_index(name="available_rows")
        )
        source_counts["source_name"] = source_counts["source_id"].map(SOURCE_MAP)
        display(source_counts)

        rng = np.random.default_rng(RANDOM_SEED)
        selected_indices = []

        for source_id in sorted(np.unique(labels_b)):
            idx = np.flatnonzero(labels_b == source_id)
            rng.shuffle(idx)
            if MAX_PER_SOURCE is not None:
                idx = idx[: min(MAX_PER_SOURCE, len(idx))]
            selected_indices.extend([int(i) for i in idx])

        rng.shuffle(selected_indices)
        print(f"Selected rows: {len(selected_indices)}")

## Evaluate Model

        This evaluates the saved GenImage detector directly on the recent dataset. No Defactify fine-tuning is performed.

In [ ]:
def image_to_pil(image_item):
            if isinstance(image_item, Image.Image):
                return image_item

            if isinstance(image_item, dict):
                if image_item.get("bytes") is not None:
                    return Image.open(io.BytesIO(image_item["bytes"]))
                if image_item.get("path"):
                    return Image.open(image_item["path"])

            if isinstance(image_item, (str, os.PathLike)):
                return Image.open(image_item)

            raise TypeError(f"Unsupported image type: {type(image_item)}")


        def preprocess_batch(pil_images):
            arr = []
            for image in pil_images:
                image = image.convert("RGB").resize((IMG_SIZE, IMG_SIZE))
                arr.append(np.asarray(image, dtype=np.float32))
            arr = np.stack(arr, axis=0)
            return tf.keras.applications.resnet50.preprocess_input(arr)


        def normalize_model_output(raw_output):
            raw = np.asarray(raw_output)
            raw = np.squeeze(raw)

            if raw.ndim == 2 and raw.shape[1] == 2:
                row_sums = raw.sum(axis=1)
                looks_like_prob = np.all(raw >= 0) and np.all(raw <= 1) and np.allclose(row_sums, 1.0, atol=1e-3)
                if looks_like_prob:
                    prob_ai = np.clip(raw[:, 1], 1e-6, 1 - 1e-6)
                    binary_logit = logit(prob_ai)
                    output_kind = "2-class probabilities"
                else:
                    prob_ai = softmax(raw, axis=1)[:, 1]
                    binary_logit = raw[:, 1] - raw[:, 0]
                    output_kind = "2-class logits"
                return prob_ai, binary_logit, output_kind

            raw = raw.reshape(-1)
            looks_like_prob = np.all(raw >= 0) and np.all(raw <= 1)
            if looks_like_prob:
                prob_ai = np.clip(raw, 1e-6, 1 - 1e-6)
                binary_logit = logit(prob_ai)
                output_kind = "1-class probability"
            else:
                binary_logit = raw.astype(float)
                prob_ai = expit(binary_logit)
                output_kind = "1-class logit"

            return prob_ai, binary_logit, output_kind


        records = []
        raw_outputs = []

        for start in tqdm(range(0, len(selected_indices), BATCH_SIZE), desc="Predicting"):
            batch_indices = selected_indices[start:start + BATCH_SIZE]
            rows = [ds[i] for i in batch_indices]
            images = [image_to_pil(row[IMAGE_COL]) for row in rows]
            batch_x = preprocess_batch(images)
            batch_raw = model.predict(batch_x, verbose=0)
            raw_outputs.append(batch_raw)

            for original_idx, row in zip(batch_indices, rows):
                source_id = int(row[SOURCE_COL])
                label_binary = int(row[LABEL_COL])
                records.append({
                    "dataset_id": DATASET_ID,
                    "split": SPLIT_TO_USE,
                    "original_index": int(original_idx),
                    "caption": row.get(CAPTION_COL, ""),
                    "label_binary": label_binary,
                    "label_binary_name": BINARY_MAP.get(label_binary, str(label_binary)),
                    "source_id": source_id,
                    "source_name": SOURCE_MAP.get(source_id, f"source_{source_id}"),
                })

        raw_outputs = np.concatenate(raw_outputs, axis=0)
        prob_ai, binary_logit, output_kind = normalize_model_output(raw_outputs)

        predictions = pd.DataFrame(records)
        predictions["prob_ai"] = prob_ai
        predictions["binary_logit"] = binary_logit
        predictions["pred_label"] = (predictions["prob_ai"] >= 0.5).astype(int)
        predictions["confidence"] = np.maximum(predictions["prob_ai"], 1 - predictions["prob_ai"])
        predictions["correct"] = predictions["pred_label"] == predictions["label_binary"].astype(int)

        if TEMPERATURE is not None:
            predictions["prob_ai_temp"] = expit(predictions["binary_logit"] / float(TEMPERATURE))
            predictions["pred_label_temp"] = (predictions["prob_ai_temp"] >= 0.5).astype(int)
            predictions["confidence_temp"] = np.maximum(predictions["prob_ai_temp"], 1 - predictions["prob_ai_temp"])

        print("Model output kind:", output_kind)
        display(predictions.head())

In [ ]:
def expected_calibration_error(y_true, prob_ai, n_bins=15):
            y_true = np.asarray(y_true, dtype=int)
            prob_ai = np.asarray(prob_ai, dtype=float)
            pred = (prob_ai >= 0.5).astype(int)
            conf = np.maximum(prob_ai, 1 - prob_ai)
            correct = (pred == y_true).astype(float)
            bins = np.linspace(0.0, 1.0, n_bins + 1)
            ece = 0.0
            for i in range(n_bins):
                if i == 0:
                    mask = (conf >= bins[i]) & (conf <= bins[i + 1])
                else:
                    mask = (conf > bins[i]) & (conf <= bins[i + 1])
                if np.any(mask):
                    ece += mask.mean() * abs(correct[mask].mean() - conf[mask].mean())
            return float(ece)


        def compute_metrics(frame, prob_col="prob_ai"):
            y_true = frame["label_binary"].astype(int).to_numpy()
            prob_ai = np.clip(frame[prob_col].astype(float).to_numpy(), 1e-6, 1 - 1e-6)
            pred = (prob_ai >= 0.5).astype(int)
            conf = np.maximum(prob_ai, 1 - prob_ai)
            wrong = pred != y_true
            tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
            return {
                "n": int(len(frame)),
                "accuracy": float(accuracy_score(y_true, pred)),
                "balanced_accuracy": float(balanced_accuracy_score(y_true, pred)),
                "auroc": float(roc_auc_score(y_true, prob_ai)) if len(np.unique(y_true)) == 2 else np.nan,
                "log_loss": float(log_loss(y_true, prob_ai, labels=[0, 1])),
                "brier": float(brier_score_loss(y_true, prob_ai)),
                "ece": expected_calibration_error(y_true, prob_ai),
                "avg_conf_wrong": float(conf[wrong].mean()) if np.any(wrong) else 0.0,
                "high_conf_errors": int(np.sum(wrong & (conf >= HIGH_CONF_THRESHOLD))),
                "high_conf_error_rate_all": float(np.mean(wrong & (conf >= HIGH_CONF_THRESHOLD))),
                "tn": int(tn),
                "fp": int(fp),
                "fn": int(fn),
                "tp": int(tp),
            }


        subset_tag = "full" if MAX_PER_SOURCE is None else f"max{MAX_PER_SOURCE}_per_source"
        prob_cols = ["prob_ai"]
        if TEMPERATURE is not None:
            prob_cols.append("prob_ai_temp")

        overall_rows = []
        for prob_col in prob_cols:
            row = compute_metrics(predictions, prob_col=prob_col)
            row["probability_column"] = prob_col
            row["temperature"] = TEMPERATURE if prob_col == "prob_ai_temp" else None
            overall_rows.append(row)

        overall_metrics = pd.DataFrame(overall_rows)
        display(overall_metrics)

        predictions_path = OUTPUT_DIR / f"defactify_predictions_{SPLIT_TO_USE}_{subset_tag}.csv"
        overall_path = OUTPUT_DIR / f"defactify_overall_metrics_{SPLIT_TO_USE}_{subset_tag}.csv"
        predictions.to_csv(predictions_path, index=False)
        overall_metrics.to_csv(overall_path, index=False)

        print("Saved predictions:", predictions_path)
        print("Saved overall metrics:", overall_path)

In [ ]:
prob_col = "prob_ai_temp" if TEMPERATURE is not None else "prob_ai"
        real_rows = predictions[predictions["source_id"] == 0]
        generator_rows = []

        for source_id in sorted(predictions["source_id"].unique()):
            source_id = int(source_id)
            if source_id == 0:
                continue
            paired = pd.concat([
                real_rows,
                predictions[predictions["source_id"] == source_id],
            ], ignore_index=True)
            metrics = compute_metrics(paired, prob_col=prob_col)
            metrics["source_id"] = source_id
            metrics["source_name"] = SOURCE_MAP.get(source_id, f"source_{source_id}")
            generator_rows.append(metrics)

        generator_metrics = pd.DataFrame(generator_rows)
        generator_path = OUTPUT_DIR / f"defactify_generator_metrics_{SPLIT_TO_USE}_{subset_tag}.csv"
        generator_metrics.to_csv(generator_path, index=False)

        display(generator_metrics[
            [
                "source_name",
                "accuracy",
                "auroc",
                "ece",
                "avg_conf_wrong",
                "high_conf_errors",
                "high_conf_error_rate_all",
                "tn",
                "fp",
                "fn",
                "tp",
            ]
        ])
        print("Saved generator metrics:", generator_path)

## Figures

In [ ]:
def save_confusion_matrix_plot(frame, prob_col, path):
            y_true = frame["label_binary"].astype(int).to_numpy()
            pred = (frame[prob_col].astype(float).to_numpy() >= 0.5).astype(int)
            cm = confusion_matrix(y_true, pred, labels=[0, 1])
            fig, ax = plt.subplots(figsize=(5, 4))
            im = ax.imshow(cm, cmap="Blues")
            ax.set_xticks([0, 1], ["Pred Real", "Pred AI"])
            ax.set_yticks([0, 1], ["True Real", "True AI"])
            ax.set_title("Defactify External Test Confusion Matrix")
            for i in range(2):
                for j in range(2):
                    ax.text(j, i, str(cm[i, j]), ha="center", va="center")
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            fig.tight_layout()
            fig.savefig(path, dpi=200)
            plt.show()


        def save_reliability_plot(frame, prob_col, path, n_bins=15):
            y_true = frame["label_binary"].astype(int).to_numpy()
            prob_ai = frame[prob_col].astype(float).to_numpy()
            bins = np.linspace(0.0, 1.0, n_bins + 1)
            xs, ys = [], []
            for i in range(n_bins):
                if i == 0:
                    mask = (prob_ai >= bins[i]) & (prob_ai <= bins[i + 1])
                else:
                    mask = (prob_ai > bins[i]) & (prob_ai <= bins[i + 1])
                if np.any(mask):
                    xs.append(prob_ai[mask].mean())
                    ys.append(y_true[mask].mean())
            fig, ax = plt.subplots(figsize=(5.5, 5))
            ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
            ax.plot(xs, ys, marker="o")
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
            ax.set_xlabel("Mean predicted probability of AI")
            ax.set_ylabel("Observed fraction AI")
            ax.set_title("Defactify External Test Reliability")
            ax.grid(alpha=0.25)
            fig.tight_layout()
            fig.savefig(path, dpi=200)
            plt.show()


        def save_confidence_histogram(frame, prob_col, path):
            y_true = frame["label_binary"].astype(int).to_numpy()
            prob_ai = frame[prob_col].astype(float).to_numpy()
            pred = (prob_ai >= 0.5).astype(int)
            conf = np.maximum(prob_ai, 1 - prob_ai)
            correct = pred == y_true
            fig, ax = plt.subplots(figsize=(7, 4.5))
            ax.hist(conf[correct], bins=20, alpha=0.70, label=f"Correct (n={correct.sum()})")
            ax.hist(conf[~correct], bins=20, alpha=0.70, label=f"Wrong (n={(~correct).sum()})")
            ax.axvline(HIGH_CONF_THRESHOLD, color="black", linestyle="--", linewidth=1)
            ax.set_xlabel("Confidence")
            ax.set_ylabel("Images")
            ax.set_title("Confidence Distribution On Defactify")
            ax.legend()
            fig.tight_layout()
            fig.savefig(path, dpi=200)
            plt.show()


        def save_generator_heatmap(metrics_frame, path):
            cols = ["accuracy", "auroc", "ece", "avg_conf_wrong", "high_conf_error_rate_all"]
            heat = metrics_frame.set_index("source_name")[cols]
            fig, ax = plt.subplots(figsize=(8, 4.5))
            im = ax.imshow(heat.to_numpy(dtype=float), aspect="auto", cmap="viridis")
            ax.set_xticks(range(len(cols)), cols, rotation=25, ha="right")
            ax.set_yticks(range(len(heat.index)), heat.index)
            ax.set_title("Generator-wise Metrics On Defactify")
            for i in range(heat.shape[0]):
                for j in range(heat.shape[1]):
                    ax.text(j, i, f"{heat.iloc[i, j]:.3f}", ha="center", va="center", color="white", fontsize=9)
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            fig.tight_layout()
            fig.savefig(path, dpi=200)
            plt.show()


        confusion_path = OUTPUT_DIR / f"defactify_confusion_matrix_{SPLIT_TO_USE}_{subset_tag}.png"
        reliability_path = OUTPUT_DIR / f"defactify_reliability_{SPLIT_TO_USE}_{subset_tag}.png"
        confidence_path = OUTPUT_DIR / f"defactify_confidence_histogram_{SPLIT_TO_USE}_{subset_tag}.png"
        heatmap_path = OUTPUT_DIR / f"defactify_generator_heatmap_{SPLIT_TO_USE}_{subset_tag}.png"

        save_confusion_matrix_plot(predictions, prob_col, confusion_path)
        save_reliability_plot(predictions, prob_col, reliability_path)
        save_confidence_histogram(predictions, prob_col, confidence_path)
        save_generator_heatmap(generator_metrics, heatmap_path)

        print("Saved figures:")
        print(confusion_path)
        print(reliability_path)
        print(confidence_path)
        print(heatmap_path)

In [ ]:
zip_path = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
        print("Created zip:", zip_path)

## Thesis Wording

        Suggested wording:

        > To assess whether the detector generalizes to newer generator distributions, the GenImage-trained ResNet-50 model was evaluated without additional fine-tuning on Defactify / MS COCOAI, a recent dataset containing real MS COCO images and synthetic images from Stable Diffusion 2.1, SDXL, Stable Diffusion 3, DALL-E 3, and Midjourney v6. This experiment is treated as an external stress test under generator distribution shift.